## Example 1: Basic checkpoint creation and retrieval with MemorySaver

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
import uuid

# Example 1: Basic checkpoint creation and retrieval with MemorySaver
memory_saver = MemorySaver()

# Create a checkpoint with state
thread_id = str(uuid.uuid4())
checkpoint_id = str(uuid.uuid4())

# Put a checkpoint
checkpoint = {
    "v": 1,
    "id": checkpoint_id,
    "ts": "2024-01-01T00:00:00Z",
    "channel_values": {"messages": ["Hello", "World"]},
    "channel_versions": {"messages": 1},
    "versions_seen": {"__pregel_tasks": {}},
    "pending_sends": []
}

config = {"configurable": {"thread_id": thread_id, "checkpoint_ns": ""}}
metadata = {"user": "test", "operation": "save"}
memory_saver.put(config, checkpoint, metadata, {})

# Get the checkpoint back
retrieved = memory_saver.get(config)
print(f"Retrieved checkpoint ID: {retrieved['id']}")
print(f"Channel values: {retrieved['channel_values']}")

## Example 2: Checkpoint metadata and filtering

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
import uuid

# Example 2: Checkpoint metadata and filtering
saver = MemorySaver()

# Create multiple checkpoints with different metadata
thread_id = str(uuid.uuid4())
for i in range(3):
    checkpoint = {
        "v": 1,
        "id": str(uuid.uuid4()),
        "ts": f"2024-01-0{i+1}T00:00:00Z",
        "channel_values": {"counter": i, "data": f"state_{i}"},
        "channel_versions": {"counter": i+1},
        "versions_seen": {"__pregel_tasks": {}},
        "pending_sends": []
    }
    config = {"configurable": {"thread_id": thread_id, "checkpoint_ns": ""}}
    metadata = {"step": i, "type": "counter", "user_id": f"user_{i}"}
    saver.put(config, checkpoint, metadata, {})

# List checkpoints with metadata
checkpoints = list(saver.list({"configurable": {"thread_id": thread_id}}))
print(f"Total checkpoints: {len(checkpoints)}")
for cp in checkpoints:
    print(f"  Step {cp.metadata.get('step')}: {cp.checkpoint['channel_values']}")

## Example 3: Memory saver with checkpoint tuple operations

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.base import Checkpoint
import uuid

# Example 3: Memory saver with checkpoint tuple operations
memory = MemorySaver()
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id, "checkpoint_ns": ""}}

# Create initial checkpoint
checkpoint1 = {
    "v": 1,
    "id": str(uuid.uuid4()),
    "ts": "2024-01-01T00:00:00Z",
    "channel_values": {"step": 0, "result": None},
    "channel_versions": {"step": 1, "result": 1},
    "versions_seen": {"__pregel_tasks": {}},
    "pending_sends": []
}
memory.put(config, checkpoint1, {"phase": "init"}, {})

# Update checkpoint with new values
checkpoint2 = {
    "v": 1,
    "id": str(uuid.uuid4()),
    "ts": "2024-01-01T00:01:00Z",
    "channel_values": {"step": 1, "result": "processed"},
    "channel_versions": {"step": 2, "result": 2},
    "versions_seen": {"__pregel_tasks": {}},
    "pending_sends": []
}
memory.put(config, checkpoint2, {"phase": "process"}, {})

# List all checkpoints for this thread
checkpoints = list(memory.list(config, limit=2))
print(f"Found {len(checkpoints)} checkpoints")
for i, cp_tuple in enumerate(checkpoints):
    vals = cp_tuple.checkpoint.get('channel_values', {})
    print(f"  Checkpoint {i}: step={vals.get('step')}, result={vals.get('result')}")

## Example 4: Working with checkpoint tuples and history

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.base import CheckpointTuple
import uuid

# Example 4: Working with checkpoint tuples and history
memory = MemorySaver()
thread_id = str(uuid.uuid4())

# Create a series of checkpoints to show history
for i in range(3):
    checkpoint = {
        "v": 1,
        "id": f"checkpoint-{i}",
        "ts": f"2024-01-01T00:0{i}:00Z",
        "channel_values": {"iteration": i, "status": f"step_{i}"},
        "channel_versions": {"iteration": i+1, "status": i+1},
        "versions_seen": {"__pregel_tasks": {}},
        "pending_sends": []
    }
    config = {"configurable": {"thread_id": thread_id, "checkpoint_ns": ""}}
    memory.put(config, checkpoint, {"run": i}, {})

# Get checkpoint history
history = list(memory.list(config))
print(f"Checkpoint history ({len(history)} checkpoints):")
for idx, checkpoint_tuple in enumerate(history):
    print(f"  {idx}: {checkpoint_tuple.checkpoint['id']}")
    
# Get specific checkpoint by config
latest = memory.get(config)
print(f"\nLatest checkpoint: {latest['id']}")

## Example 5: Working with checkpoint metadata

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.base import CheckpointMetadata
import uuid

# Example 5: Working with checkpoint metadata
memory = MemorySaver()
thread_id = str(uuid.uuid4())

# Create checkpoints with rich metadata
checkpoints_data = [
    {"task": "initialize", "priority": "high", "user": "admin"},
    {"task": "process", "priority": "medium", "user": "worker"},
    {"task": "finalize", "priority": "low", "user": "system"}
]

for i, meta in enumerate(checkpoints_data):
    checkpoint = {
        "v": 1,
        "id": f"meta-checkpoint-{i}",
        "ts": f"2024-01-01T00:{i:02d}:00Z",
        "channel_values": {"phase": meta["task"], "index": i},
        "channel_versions": {"phase": i+1, "index": i+1},
        "versions_seen": {"__pregel_tasks": {}},
        "pending_sends": []
    }
    config = {"configurable": {"thread_id": thread_id, "checkpoint_ns": ""}}
    memory.put(config, checkpoint, meta, {})

# Query checkpoints with metadata
all_checkpoints = list(memory.list(config))
print(f"Total checkpoints with metadata: {len(all_checkpoints)}")
for cp_tuple in all_checkpoints[:2]:  # Show first 2
    print(f"  Task: {cp_tuple.metadata.get('task')}, Priority: {cp_tuple.metadata.get('priority')}")